In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as utils
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.decomposition import PCA

from dataloader import build_smri_tensor
from model.smriEncoders import SMRIEncoder, SMRIAttentionEncoder

C:\Users\Lenovo\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
# mlp or attention
model_type = "mlp"

config = {
        "data": {
            "time_seires": "abide.npy",
            "time_series_subjects_order": "subject_order.txt",
            "smri": "abide_smri.csv",
            "train_set": 0.7,
            "val_set": 0.1,
            "batch_size": 16
        },
        "train_mlp": {
            "epochs": 10,
            "lr": 5e-5
        },
        "train_attention": {
            "epochs": 20,
            "lr": 3e-5
        }
    }

In [3]:
class SMRIClassifier(nn.Module):

    def __init__(self, smri_dim):
        super().__init__()

        if model_type == "mlp":
            self.encoder = SMRIEncoder(smri_dim)
        elif model_type == "attention":
            self.encoder = SMRIAttentionEncoder(smri_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Dropout(0.5),
            nn.Linear(32, 2)
        )

    def forward(self, x):

        feat = self.encoder(x)   # (B,64)

        out = self.classifier(feat)

        return out

In [4]:
# -----------------------------
# Seed
# -----------------------------
def set_seed(seed=21):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
# -----------------------------
# Eval
# -----------------------------
@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    preds = []
    probs = []
    labels = []

    for smri, y in loader:

        smri = smri.to(device)
        y = y.long().to(device)

        out = model(smri)

        prob = torch.softmax(out, dim=1)[:, 1]
        pred = torch.argmax(out, dim=1)

        preds.extend(pred.cpu().numpy())
        probs.extend(prob.cpu().numpy())
        labels.extend(y.cpu().numpy())

    acc = accuracy_score(labels, preds)

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.5

    return acc, auc

In [6]:
# -----------------------------
# Main
# -----------------------------
def main():

    set_seed(21)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


    # -------------------------
    # Load labels from abide.npy
    # -------------------------
    data = np.load(
        config["data"]["time_seires"],
        allow_pickle=True
    ).item()

    labels = torch.from_numpy(
        data["label"]
    ).long()

    num_subjects = len(labels)

    # -------------------------
    # Load sMRI
    # -------------------------
    smri_tensor, smri_dim = build_smri_tensor(
        config["data"],
        num_subjects
    )

    dataset = utils.TensorDataset(
        smri_tensor,
        labels
    )

    train_length = int(
        num_subjects * config["data"]["train_set"]
    )

    val_length = int(
        num_subjects * config["data"]["val_set"]
    )

    test_length = (
        num_subjects
        - train_length
        - val_length
    )

    generator = torch.Generator().manual_seed(42)

    train_dataset, val_dataset, test_dataset = \
        torch.utils.data.random_split(
            dataset,
            [train_length, val_length, test_length],
            generator=generator
        )
        
        
    # -------------------------
    # PCA
    # -------------------------

    pca_dim = 256

    train_idx = train_dataset.indices
    val_idx = val_dataset.indices
    test_idx = test_dataset.indices

    X_train = smri_tensor[train_idx].numpy()
    X_val   = smri_tensor[val_idx].numpy()
    X_test  = smri_tensor[test_idx].numpy()

    pca = PCA(
        n_components=0.95,
        random_state=21
    )
    

    X_train = pca.fit_transform(X_train)
    
    actual_pca_dim = X_train.shape[1]

    X_val = pca.transform(X_val)
    X_test = pca.transform(X_test)

    X_train = torch.FloatTensor(X_train)
    X_val   = torch.FloatTensor(X_val)
    X_test  = torch.FloatTensor(X_test)
    
    
    y_train = labels[train_idx]
    y_val   = labels[val_idx]
    y_test  = labels[test_idx]

    train_dataset = utils.TensorDataset(
        X_train,
        y_train
    )

    val_dataset = utils.TensorDataset(
        X_val,
        y_val
    )

    test_dataset = utils.TensorDataset(
        X_test,
        y_test
    )
        
                

    train_loader = utils.DataLoader(
        train_dataset,
        batch_size=config["data"]["batch_size"],
        shuffle=True
    )

    val_loader = utils.DataLoader(
        val_dataset,
        batch_size=config["data"]["batch_size"],
        shuffle=False
    )

    test_loader = utils.DataLoader(
        test_dataset,
        batch_size=config["data"]["batch_size"],
        shuffle=False
    )

    model = SMRIClassifier(
        actual_pca_dim
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    
    
    if model_type == "mlp":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["train_mlp"]["lr"],
            weight_decay=1e-3
        )
        epochs = config["train_mlp"]["epochs"]
    elif model_type == "attention":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["train_attention"]["lr"],
            weight_decay=1e-4
        )
        epochs = config["train_attention"]["epochs"]

    best_val_acc = 0

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for smri, y in train_loader:

            smri = smri.to(device)
            y = y.long().to(device)

            optimizer.zero_grad()

            out = model(smri)

            loss = criterion(out, y)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        val_acc, val_auc = evaluate(
            model,
            val_loader,
            device
        )

        test_acc, test_auc = evaluate(
            model,
            test_loader,
            device
        )
        
        train_acc, train_auc = evaluate(
            model,
            train_loader,
            device
        )

        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss {train_loss:.4f} | "
            f"Train ACC {train_acc:.4f} | "
            f"Train AUC {train_auc:.4f} | "
            f"Val ACC {val_acc:.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Test ACC {test_acc:.4f} | "
            f"Test AUC {test_auc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                "best_smri_model.pth"
            )

    print(f"\nBest Val ACC = {best_val_acc:.4f}")
    model.load_state_dict(
        torch.load("best_smri_model.pth")
    )
    train_acc, train_auc = evaluate(
        model,
        train_loader,
        device
    )

    val_acc, val_auc = evaluate(
        model,
        val_loader,
        device
    )

    test_acc, test_auc = evaluate(
        model,
        test_loader,
        device
    )
    
    print("\n" + "=" * 60)
    print("BEST MODEL RESULTS")
    print("=" * 60)

    print(
        f"TRAIN : ACC={train_acc:.4f} | AUC={train_auc:.4f}"
    )

    print(
        f"VAL   : ACC={val_acc:.4f} | AUC={val_auc:.4f}"
    )

    print(
        f"TEST  : ACC={test_acc:.4f} | AUC={test_auc:.4f}"
    )

    print("=" * 60)


if __name__ == "__main__":
    main()

Epoch 001 | Loss 0.7033 | Train ACC 0.4731 | Train AUC 0.5040 | Val ACC 0.4400 | Val AUC 0.4290 | Test ACC 0.4975 | Test AUC 0.4753
Epoch 002 | Loss 0.6996 | Train ACC 0.4887 | Train AUC 0.5338 | Val ACC 0.4500 | Val AUC 0.4354 | Test ACC 0.4975 | Test AUC 0.4868
Epoch 003 | Loss 0.6901 | Train ACC 0.5170 | Train AUC 0.5622 | Val ACC 0.4600 | Val AUC 0.4226 | Test ACC 0.5172 | Test AUC 0.4967
Epoch 004 | Loss 0.6999 | Train ACC 0.5382 | Train AUC 0.5916 | Val ACC 0.4600 | Val AUC 0.4278 | Test ACC 0.5172 | Test AUC 0.4912
Epoch 005 | Loss 0.6956 | Train ACC 0.5595 | Train AUC 0.6159 | Val ACC 0.4600 | Val AUC 0.4330 | Test ACC 0.5172 | Test AUC 0.4993
Epoch 006 | Loss 0.6904 | Train ACC 0.5921 | Train AUC 0.6440 | Val ACC 0.4500 | Val AUC 0.4422 | Test ACC 0.5271 | Test AUC 0.5027
Epoch 007 | Loss 0.6855 | Train ACC 0.6147 | Train AUC 0.6726 | Val ACC 0.4800 | Val AUC 0.4426 | Test ACC 0.5271 | Test AUC 0.5147
Epoch 008 | Loss 0.6898 | Train ACC 0.6331 | Train AUC 0.7002 | Val ACC 0.50